# Input adapters

All inputs below represent the same paired synthetic observations. This notebook performs no biological preprocessing.

In [1]:
import coati as ct
import numpy as np
import pandas as pd
from pathlib import Path
import tempfile

data = ct.datasets.paired(n=40)
work = Path(tempfile.mkdtemp(prefix="coati-inputs-"))

## Arrays and an ordered dictionary

In [2]:
arrays = ct.TemporalData(data.primary, data.times, data.secondary)
mapping = ct.TemporalData.from_dict(dict(zip(["early", "middle", "late"], data.primary)), times=data.times)
np.testing.assert_array_equal(mapping.primary[1], data.primary[1])

## NPZ

In [3]:
keys = data.save_npz(work)
loaded = ct.TemporalData.from_npz(work / "primary.npz", secondary_path=work / "secondary.npz", keys=keys, times=data.times)
np.testing.assert_array_equal(loaded.secondary[1], data.secondary[1])

## DataFrame and CSV

In [4]:
frames = []
for t, x, y in zip(data.times, data.primary, data.secondary):
    frames.append(pd.DataFrame({"time": t, "PC1": x[:, 0], "PC2": x[:, 1], "LSI1": y[:, 0], "LSI2": y[:, 1], "LSI3": y[:, 2]}))
frame = pd.concat(frames, ignore_index=True)
frame.to_csv(work / "paired.csv", index=False)
converted = ct.TemporalData.from_csv(work / "paired.csv", time_key="time", feature_keys=["PC1", "PC2"], secondary_keys=["LSI1", "LSI2", "LSI3"])
np.testing.assert_allclose(converted.primary[1], data.primary[1], atol=1e-7)
frame.head()

,time,PC1,PC2,LSI1,LSI2,LSI3
0,0.0,0.207544,0.292074,0.207544,0.292074,0.144562
1,0.0,0.238425,0.306294,0.238425,0.306294,0.167608
2,0.0,0.167860,0.321696,0.167860,0.321696,0.127900
3,0.0,0.278240,0.356825,0.278240,0.356825,0.207207
4,0.0,0.157776,0.224075,0.157776,0.224075,0.092351


## AnnData and h5ad

Install the `anndata` extra before this cell. Both embeddings share cell identities in `obs`.

In [5]:
import anndata as ad
adata = ad.AnnData(np.concatenate(data.primary), obs=pd.DataFrame({"time": np.repeat(data.times, 40)}, index=[str(i) for i in range(120)]))
adata.obsm["X_lsi"] = np.concatenate(data.secondary)
adata.write_h5ad(work / "paired.h5ad")
converted = ct.TemporalData.from_h5ad(work / "paired.h5ad", time_key="time", secondary_obsm_key="X_lsi")
np.testing.assert_array_equal(converted.secondary[1], data.secondary[1])
print("All adapters preserve the supplied data.")

All adapters preserve the supplied data.
